<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/10_serving_architecture.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/10_serving_architecture.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# 10 — Serving Architecture: Batching, Caching, Canary

**Objectives**
- Quantify the latency/throughput tradeoff of dynamic batching.
- Size a cache from the hit-rate curve.
- Use an error budget to decide canary promotion vs rollback.


In [ ]:
def simulate_batch(
    batch_size: int, base_ms: float = 20, per_item_ms: float = 2.0, qps: float = 500
):
    # Larger batches amortize per-request overhead but add queueing delay
    compute_ms = base_ms + per_item_ms * batch_size
    queue_wait_ms = (batch_size / qps) * 1000 / 2
    throughput = batch_size / (compute_ms / 1000)
    return {
        "batch_size": batch_size,
        "latency_ms": compute_ms + queue_wait_ms,
        "throughput_rps": throughput,
    }


results = [simulate_batch(b) for b in (1, 4, 8, 16, 32, 64)]
for r in results:
    print(
        f"batch={r['batch_size']:>2} latency={r['latency_ms']:7.1f}ms throughput={r['throughput_rps']:8.1f} rps"
    )

best = max((r for r in results if r["latency_ms"] <= 300), key=lambda r: r["throughput_rps"])
print("optimal_batch_size:", best["batch_size"])

In [ ]:
# Cache sizing: effective backend load falls with hit rate
qps = 500
for hit_rate in (0.0, 0.5, 0.8, 0.9, 0.95):
    backend_qps = qps * (1 - hit_rate)
    print(f"hit_rate={hit_rate:.2f} backend_qps={backend_qps:6.1f}")

hit_rate = 0.9
print("backend_qps_at_90pct:", qps * (1 - hit_rate))

In [ ]:
# Canary decision from an error budget
error_budget = 0.001  # 0.1% of requests may fail
canary_requests = 50_000
canary_errors = 80

observed_error_rate = canary_errors / canary_requests
budget_consumed = observed_error_rate / error_budget
print(f"observed_error_rate: {observed_error_rate:.5f}")
print(f"error_budget_consumed: {budget_consumed:.2f}x")
print("decision:", "promote" if budget_consumed <= 1.0 else "rollback")

**Architect's notes**
- Dynamic batching maximizes GPU utilization but every millisecond of queue wait is
  paid by the user; pick the knee of the curve, not the maximum batch.
- A cache is a latency lever and a consistency risk: define invalidation first.
- Canaries are cheap insurance; automate the promote/rollback decision with budgets.
